In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
import os
import logging
from tensorflow.keras.models import Sequential, clone_model, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy, AUC
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc

In [2]:
# Set seed for reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [3]:
# Load the dataset
data_x = pd.read_csv('X_mkt.csv')
data_y = pd.read_csv('y_mkt.csv')

In [4]:
# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in data_x.columns:
    data_x = data_x.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in data_y.columns:
    data_y = data_y.drop(columns=['Unnamed: 0'])

In [5]:
# Convert to numpy arrays
X = data_x.values
y = data_y['Target'].values

In [6]:
# Define sliding window parameters
window_size = 600
test_gap = 180
step_size = 1

In [7]:
# Determine the start point to process only the last 500 windows
start_point = len(X) - (500 * step_size) - window_size - test_gap

In [8]:
# Lists to store results
all_test_predictions = []
all_actual_test_values = []
all_prediction_probabilities = []
all_predicted_classes = []

In [9]:
# Define model creation function
def create_model(hu=20, lookback=10, features=1):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=hu, input_shape=(lookback, features), activation='relu', return_sequences=False, name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))
    model.add(Dense(units=1, activation='sigmoid', name='Output'))
    opt = Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0)
    model.compile(optimizer=opt, loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    return model

In [10]:
# Initialize and save the initial model
initial_model = create_model(hu=20, lookback=10, features=X.shape[1])
initial_model.compile(optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), loss=BinaryCrossentropy(), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
initial_model.save('initial_model_HML.h5')

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [11]:
# Save the weights of the initial model
initial_model.save_weights('current_model_MKT.weights.h5')

In [12]:
# EarlyStopping callback
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

In [13]:
# Checkpoint callback
checkpoint = ModelCheckpoint(filepath='model_checkpoint_MKT.weights.h5', save_best_only=True, save_weights_only=True, monitor='loss', mode='min')

In [14]:
# Train and evaluate the model with sliding window
for start in range(start_point, len(X) - window_size - test_gap, step_size):
    try:
        end = start + window_size
        X_train_window = X[start:end]
        y_train_window = y[start:end]
        
        # Define test indices
        test_start = end + test_gap
        test_end = test_start + 1
        if test_end > len(X):
            break
        X_test_window = X[test_start:test_end]
        y_test_window = y[test_start:test_end]

        # Sequence length for training
        seqlen = 10
        # Number of features
        numfeat = X_train_window.shape[1]

        # Generate sequence input for the training dataset
        train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=32)

        # Load and clone the initial model for the first iteration, then the previously trained model for subsequent iterations
        model = create_model(hu=20, lookback=10, features=numfeat)
        model.load_weights('current_model_MKT.weights.h5')

        model_clone = clone_model(model)
        model_clone.set_weights(model.get_weights())

        model_transfer = Sequential(model_clone.layers[:-1])
        model_transfer.add(Dense(1, activation='sigmoid'))

        # Freeze the cloned layers initially
        for layer in model_transfer.layers[:-1]:
            layer.trainable = False

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=4, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

        # Unfreeze the cloned layers for further training
        for layer in model_transfer.layers[:-1]:
            layer.trainable = True

        model_transfer.compile(loss=BinaryCrossentropy(), optimizer=Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
        model_transfer.fit(train_generator, epochs=70, verbose=1, class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_window), y=y_train_window))), shuffle=False, callbacks=[early_stopping])

        # Save the best model during training for the current window
        model_transfer.save('current_model_MKT.h5')
        model_transfer.save_weights('current_model_MKT.weights.h5')

        # Predict on the single test sample directly
        test_input = np.expand_dims(X_test_window, axis=0)
        prediction = model_transfer.predict(test_input)
        prediction_probability = prediction[0][0]
        predicted_class = int(prediction_probability > 0.5)
        print(f'Test Prediction: {prediction_probability:.2f} (Class: {predicted_class})')

        # Store test predictions, actual values, and predicted classes
        all_test_predictions.append(prediction_probability)
        all_actual_test_values.append(y_test_window[0])
        all_prediction_probabilities.append(prediction_probability)
        all_predicted_classes.append(predicted_class)

    except Exception as e:
        print(f'Error at window {start} to {end}: {e}')
        break


Epoch 1/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5727 - auc_1: 0.5234 - loss: 11.5332
Epoch 2/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5789 - auc_1: 0.5272 - loss: 10.9596
Epoch 3/4
 1/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8750 - auc_1: 0.6917 - loss: 1.7627

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5958 - auc_1: 0.5756 - loss: 9.1870 
Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6393 - auc_1: 0.6219 - loss: 7.5691 
Epoch 1/70


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4055 - auc_2: 0.3692 - loss: 1.5426
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5130 - auc_2: 0.5901 - loss: 3.4416 
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4460 - auc_2: 0.3883 - loss: 6.1721 
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5175 - auc_2: 0.5807 - loss: 1.7798 
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4296 - auc_2: 0.4550 - loss: 1.7140 
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4373 - auc_2: 0.4798 - loss: 0.9558 
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3914 - auc_2: 0.4389 - loss: 0.9678 
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4172 - auc_2: 0.4573 - loss: 0.7952 
Epoch 9/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3821 - auc_2: 0.4328 - loss: 0.8188 
Epoch 10/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4178 - auc_2: 0.4244 - loss: 0.7364 
E

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Test Prediction: 0.52 (Class: 1)
Epoch 1/4


... (3,489 outputs left out)

Epoch 4/4
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_1: 0.2750 - loss: 0.5160     
Epoch 1/70


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5150    
Epoch 2/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5132     
Epoch 3/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5135     
Epoch 4/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5138     
Epoch 5/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5141     
Epoch 6/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5145     
Epoch 7/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5148     
Epoch 8/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.2750 - loss: 0.5151     
Epoch 9/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9351 - auc_2: 0.3185 - loss: 0.5155     
Epoch 10/70
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.93

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Test Prediction: 0.51 (Class: 1)


In [15]:
# Save the results to files for further analysis
results = {
    "test_predictions": all_test_predictions,
    "actual_test_values": all_actual_test_values,
    "prediction_probabilities": all_prediction_probabilities,
    "predicted_classes": all_predicted_classes
}

In [16]:
results_df = pd.DataFrame(results)
results_df.to_csv('TF_results_MKT_ADAM.csv', index=False)

In [19]:
# Generate and print the combined classification report
combined_classification_report = classification_report(all_actual_test_values, all_predicted_classes, output_dict=True)
print("Combined Classification Report:")
print(combined_classification_report)

Combined Classification Report:
{'0': {'precision': 0.48138297872340424, 'recall': 0.6013289036544851, 'f1-score': 0.534711964549483, 'support': 301.0}, '1': {'precision': 0.03225806451612903, 'recall': 0.020100502512562814, 'f1-score': 0.02476780185758514, 'support': 199.0}, 'accuracy': 0.37, 'macro avg': {'precision': 0.25682052161976665, 'recall': 0.31071470308352395, 'f1-score': 0.27973988320353405, 'support': 500.0}, 'weighted avg': {'precision': 0.3026312628689087, 'recall': 0.37000000000000005, 'f1-score': 0.3317541877981076, 'support': 500.0}}


In [18]:
# Save the final model
model_transfer.save('final_model_MKT.h5')